In [58]:
import pandas as pd
from pathlib import Path

# ----------------------------------------------------------
# Base directories
# ----------------------------------------------------------
BASE    = Path("/Users/davekokel/Projects/carp_v2")
RAW     = BASE / "seed_kits" / "legacy_wrangling" / "raw"
WORKING = BASE / "seed_kits" / "legacy_wrangling" / "working"

# ----------------------------------------------------------
# Explicit paths to each raw data file
# ----------------------------------------------------------
roi_paths_path     = RAW / "2025-11-13-092338-korra_aang_roi_root_tiffs_good-3.xlsx"
imaging_sheet_path = RAW / "2025-11-13-124226-imaging_sheet.xlsx"
inj_plasmid_path   = RAW / "Unique_injected_plasmid__preview_dqm.xlsx"
inj_rna_path       = RAW / "Unique_injected_rna__preview_dqm.xlsx"
parent_map_path    = RAW / "Unique_parent_names__mom_dad_combined__preview_dqm.xlsx"
constructs_path    = RAW / "constructs_plasmid.csv"

# ----------------------------------------------------------
# Load DataFrames
# ----------------------------------------------------------
df_roi        = pd.read_excel(roi_paths_path)
df_sheet      = pd.read_excel(imaging_sheet_path)
df_inj_plasm  = pd.read_excel(inj_plasmid_path)
df_inj_rna    = pd.read_excel(inj_rna_path)
df_parent_map = pd.read_excel(parent_map_path)
df_constructs = pd.read_csv(constructs_path)

print("df_roi:", df_roi.shape)
print("df_sheet:", df_sheet.shape)
print("df_inj_plasm:", df_inj_plasm.shape)
print("df_inj_rna:", df_inj_rna.shape)
print("df_parent_map:", df_parent_map.shape)
print("df_constructs:", df_constructs.shape)

df_roi.head(), df_sheet.head()

df_roi: (976, 7)
df_sheet: (306, 20)
df_inj_plasm: (25, 2)
df_inj_rna: (44, 2)
df_parent_map: (41, 3)
df_constructs: (287, 12)


(        date_experiment         fish               roi_rel  \
 0  20250428_mem_histone  fish1_72hpf             roi1_tail   
 1  20250428_mem_histone  fish1_72hpf  roi2_hindbrain_spine   
 2  20250429_mem_cytosol  fish1_24hpf                  roi1   
 3  20250429_mem_cytosol  fish1_24hpf             roi1_test   
 4  20250429_mem_cytosol  fish2_48hpf                  roi1   
 
                roi_name  roi_tiffs  \
 0             roi1_tail       3206   
 1  roi2_hindbrain_spine       9760   
 2                  roi1       6873   
 3             roi1_test         46   
 4                  roi1       3180   
 
                                              roi_dir dataset  
 0  /clusterfs/vast/abcabc/Korra_Foundation/202504...   korra  
 1  /clusterfs/vast/abcabc/Korra_Foundation/202504...   korra  
 2  /clusterfs/vast/abcabc/Korra_Foundation/202504...   korra  
 3  /clusterfs/vast/abcabc/Korra_Foundation/202504...   korra  
 4  /clusterfs/vast/abcabc/Korra_Foundation/202504...   korra  ,

In [32]:
def extract_run_key_from_sheet(path: str | float) -> str | None:
    if not isinstance(path, str) or not path.strip():
        return None
    s = path.replace("\\", "/").strip()
    low = s.lower()
    if "abcabc/" in low:
        s = s.split("abcabc/", 1)[1]
    parts = s.split("/")
    if len(parts) >= 2:
        return "/".join(parts[:2])
    return s

def extract_run_key_from_roi(path: str | float) -> str | None:
    if not isinstance(path, str) or not path.strip():
        return None
    s = path.replace("\\", "/").strip()
    low = s.lower()
    if "abcabc/" in low:
        s = s.split("abcabc/", 1)[1]
    parts = s.split("/")
    if len(parts) >= 2:
        return "/".join(parts[:2])
    return s

df_sheet["run_key"] = df_sheet["Data location"].apply(extract_run_key_from_sheet)
df_roi["run_key"]   = df_roi["roi_dir"].apply(extract_run_key_from_roi)

joined = df_roi.merge(
    df_sheet,
    how="left",
    on="run_key",
    suffixes=("_roi", "_sheet"),
)

joined = joined.drop_duplicates(subset=["roi_dir"]).reset_index(drop=True)

print("ROIs:", len(df_roi), "  joined:", len(joined))
joined.head()

ROIs: 976   joined: 976


,date_experiment,fish,roi_rel,roi_name,roi_tiffs,roi_dir,dataset,run_key,date_mount,mount_id,...,Mounting Orientation,Date screened/Initial feedback,Date imaged,Time placed in scope,Start of imaging time,End of imaging time,Imaged Locations,Unique Targets with blanks,Unique Targets,Data location
0,20250428_mem_histone,fish1_72hpf,roi1_tail,roi1_tail,3206,/clusterfs/vast/abcabc/Korra_Foundation/202504...,korra,Korra_Foundation/20250428_mem_histone,2025-04-28,NaN,...,DT,2025-04-27 00:00:00,2025-04-28,12:30:00,14:30:00,NaN,"spinal_cord, skin, muscle, tail, notochord, hi...",nuclei,nuclei,X:\abcabc\Korra_Foundation\20250428_mem_histon...
1,20250428_mem_histone,fish1_72hpf,roi2_hindbrain_spine,roi2_hindbrain_spine,9760,/clusterfs/vast/abcabc/Korra_Foundation/202504...,korra,Korra_Foundation/20250428_mem_histone,2025-04-28,NaN,...,DT,2025-04-27 00:00:00,2025-04-28,12:30:00,14:30:00,NaN,"spinal_cord, skin, muscle, tail, notochord, hi...",nuclei,nuclei,X:\abcabc\Korra_Foundation\20250428_mem_histon...
2,20250429_mem_cytosol,fish1_24hpf,roi1,roi1,6873,/clusterfs/vast/abcabc/Korra_Foundation/202504...,korra,Korra_Foundation/20250429_mem_cytosol,2025-05-01,NaN,...,DT,2025-04-30 00:00:00,2025-05-01,NaN,NaN,NaN,"hindbrain, spinal_cord, skin, muscle, notochor...",cytosol,cytosol,X:\abcabc\Korra_Foundation\20250429_mem_cytoso...
3,20250429_mem_cytosol,fish1_24hpf,roi1_test,roi1_test,46,/clusterfs/vast/abcabc/Korra_Foundation/202504...,korra,Korra_Foundation/20250429_mem_cytosol,2025-05-01,NaN,...,DT,2025-04-30 00:00:00,2025-05-01,NaN,NaN,NaN,"hindbrain, spinal_cord, skin, muscle, notochor...",cytosol,cytosol,X:\abcabc\Korra_Foundation\20250429_mem_cytoso...
4,20250429_mem_cytosol,fish2_48hpf,roi1,roi1,3180,/clusterfs/vast/abcabc/Korra_Foundation/202504...,korra,Korra_Foundation/20250429_mem_cytosol,2025-05-01,NaN,...,DT,2025-04-30 00:00:00,2025-05-01,NaN,NaN,NaN,"hindbrain, spinal_cord, skin, muscle, notochor...",cytosol,cytosol,X:\abcabc\Korra_Foundation\20250429_mem_cytoso...


In [33]:
df = joined.copy()

df["date_born"]      = pd.to_datetime(df["Date born"], errors="coerce").dt.date
df["parent_female"]  = df["ZF female genotype"]
df["parent_male"]    = df["ZF male genotype"]

print("Non-null date_born:", df["date_born"].notna().sum())
print("Non-null parent_female:", df["parent_female"].notna().sum())
print("Non-null parent_male:", df["parent_male"].notna().sum())

df[["roi_dir", "date_experiment", "fish", "parent_female", "parent_male"]].head(10)

Non-null date_born: 816
Non-null parent_female: 786
Non-null parent_male: 804


,roi_dir,date_experiment,fish,parent_female,parent_male
0,/clusterfs/vast/abcabc/Korra_Foundation/202504...,20250428_mem_histone,fish1_72hpf,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301)
1,/clusterfs/vast/abcabc/Korra_Foundation/202504...,20250428_mem_histone,fish1_72hpf,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301)
2,/clusterfs/vast/abcabc/Korra_Foundation/202504...,20250429_mem_cytosol,fish1_24hpf,membrane Halo,membrane Halo
3,/clusterfs/vast/abcabc/Korra_Foundation/202504...,20250429_mem_cytosol,fish1_24hpf,membrane Halo,membrane Halo
4,/clusterfs/vast/abcabc/Korra_Foundation/202504...,20250429_mem_cytosol,fish2_48hpf,membrane Halo,membrane Halo
5,/clusterfs/vast/abcabc/Korra_Foundation/202504...,20250429_mem_cytosol,fish3_mem-halo_24hpf,membrane Halo,membrane Halo
6,/clusterfs/vast/abcabc/Korra_Foundation/202504...,20250429_mem_cytosol,fish3_mem-halo_24hpf,membrane Halo,membrane Halo
7,/clusterfs/vast/abcabc/Korra_Foundation/202504...,20250429_mem_cytosol,fish4_mem-halo_48hpf,membrane Halo,membrane Halo
8,/clusterfs/vast/abcabc/Korra_Foundation/202504...,20250429_mem_cytosol,fish4_mem-halo_48hpf,membrane Halo,membrane Halo
9,/clusterfs/vast/abcabc/Korra_Foundation/202504...,20250429_mem_cytosol,fish4_mem-halo_48hpf,membrane Halo,membrane Halo


In [34]:
#df.columns.tolist()

In [35]:
import pandas as pd  # ensure imported

# 1) Start from df and drop any old mapping columns to avoid _x/_y chaos
mapping_prefixes = [
    "parent_fish_name_female",
    "parent_fish_name_male",
    "plasmid_base_code_female",
    "plasmid_base_code_male",
    "allele_female",
    "allele_male",
]
to_drop = [c for c in df.columns if any(c.startswith(p) for p in mapping_prefixes)]
df = df.drop(columns=to_drop, errors="ignore")

# 2) Build a clean parent map → base_code + allele
pm = df_parent_map.copy()
pm.columns = [c.strip().lower() for c in pm.columns]

parent_name_col = "parent_fish_name"
base_col        = "plasmid_base_code"
allele_col      = "allele"

pm_keyed = pm[[parent_name_col, base_col, allele_col]].dropna(subset=[parent_name_col])
pm_keyed[parent_name_col] = pm_keyed[parent_name_col].astype(str).str.strip()

# 3) Join female parent mapping
df = df.merge(
    pm_keyed.add_suffix("_female"),
    how="left",
    left_on="parent_female",
    right_on=f"{parent_name_col}_female",
)

# 4) Join male parent mapping
df = df.merge(
    pm_keyed.add_suffix("_male"),
    how="left",
    left_on="parent_male",
    right_on=f"{parent_name_col}_male",
)

# 5) Extract genotype base codes / alleles
df["genotype_female_base_code"] = df[f"{base_col}_female"]
df["genotype_female_allele"]    = df[f"{allele_col}_female"]
df["genotype_male_base_code"]   = df[f"{base_col}_male"]
df["genotype_male_allele"]      = df[f"{allele_col}_male"]

def combine_codes(row: pd.Series) -> str | None:
    codes: list[str] = []
    if isinstance(row["genotype_female_base_code"], str):
        codes.append(row["genotype_female_base_code"])
    if isinstance(row["genotype_male_base_code"], str):
        codes.append(row["genotype_male_base_code"])
    return ",".join(sorted(set(codes))) if codes else None

def combine_alleles(row: pd.Series) -> str | None:
    parts: list[str] = []
    if isinstance(row["genotype_female_base_code"], str) and pd.notna(row["genotype_female_allele"]):
        parts.append(f"{row['genotype_female_base_code']}:{row['genotype_female_allele']}")
    if isinstance(row["genotype_male_base_code"], str) and pd.notna(row["genotype_male_allele"]):
        parts.append(f"{row['genotype_male_base_code']}:{row['genotype_male_allele']}")
    return ",".join(parts) if parts else None

df["genotype_base_codes"]   = df.apply(combine_codes, axis=1)
df["genotype_allele_codes"] = df.apply(combine_alleles, axis=1)

# 6) Pretty genotype label from parent_female / parent_male
def _norm_parent(val) -> str:
    if isinstance(val, str):
        return val.strip()
    if pd.isna(val):
        return ""
    return str(val).strip()

def make_genotype_pretty(row: pd.Series) -> str | None:
    f = _norm_parent(row.get("parent_female"))
    m = _norm_parent(row.get("parent_male"))
    if f and m:
        return f"{f} × {m}"
    if f:
        return f
    if m:
        return m
    return None

df["genotype_pretty"] = df.apply(make_genotype_pretty, axis=1)

df[[
    "roi_dir", "parent_female", "parent_male",
    "genotype_base_codes", "genotype_allele_codes", "genotype_pretty"
]].head(15)

,roi_dir,parent_female,parent_male,genotype_base_codes,genotype_allele_codes,genotype_pretty
0,/clusterfs/vast/abcabc/Korra_Foundation/202504...,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),pDQM005,"pDQM005:301,pDQM005:301",ef1a:2xLynk:tdmSG(J) (F2 of allele 301) × ef1a...
1,/clusterfs/vast/abcabc/Korra_Foundation/202504...,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),pDQM005,"pDQM005:301,pDQM005:301",ef1a:2xLynk:tdmSG(J) (F2 of allele 301) × ef1a...
2,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,membrane Halo,Swinburne,"Swinburne:Swinburne,Swinburne:Swinburne",membrane Halo × membrane Halo
3,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,membrane Halo,Swinburne,"Swinburne:Swinburne,Swinburne:Swinburne",membrane Halo × membrane Halo
4,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,membrane Halo,Swinburne,"Swinburne:Swinburne,Swinburne:Swinburne",membrane Halo × membrane Halo
5,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,membrane Halo,Swinburne,"Swinburne:Swinburne,Swinburne:Swinburne",membrane Halo × membrane Halo
6,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,membrane Halo,Swinburne,"Swinburne:Swinburne,Swinburne:Swinburne",membrane Halo × membrane Halo
7,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,membrane Halo,Swinburne,"Swinburne:Swinburne,Swinburne:Swinburne",membrane Halo × membrane Halo
8,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,membrane Halo,Swinburne,"Swinburne:Swinburne,Swinburne:Swinburne",membrane Halo × membrane Halo
9,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,membrane Halo,Swinburne,"Swinburne:Swinburne,Swinburne:Swinburne",membrane Halo × membrane Halo


In [36]:
cons = df_constructs.copy()
cons["plasmid_base_code"] = cons["plasmid_code"].astype(str).str.strip()

marker_cols = ["plasmid_base_code", "fluor_code", "tag_code", "tag_pos"]
cons_small = cons[marker_cols].dropna(subset=["plasmid_base_code"])

df["genotype_base_list"] = (
    df["genotype_base_codes"]
    .dropna()
    .astype(str)
    .str.split(",")
)

gexp = (
    df[["roi_dir", "genotype_base_list"]]
    .explode("genotype_base_list")
    .rename(columns={"genotype_base_list": "plasmid_base_code"})
)

gexp["plasmid_base_code"] = gexp["plasmid_base_code"].astype(str).str.strip()

gjoin = gexp.merge(cons_small, how="left", on="plasmid_base_code")

def agg_uniq(series):
    vals = [v for v in series.dropna().astype(str) if v.strip()]
    return ",".join(sorted(set(vals))) if vals else None

g_per_roi = (
    gjoin.groupby("roi_dir", as_index=False)
    .agg(
        genotype_marker_fluor_codes=("fluor_code", agg_uniq),
        genotype_marker_tag_codes=("tag_code", agg_uniq),
    )
)

df = df.merge(g_per_roi, how="left", on="roi_dir")

df[[
    "roi_dir", "genotype_base_codes",
    "genotype_marker_fluor_codes", "genotype_marker_tag_codes"
]].head(20)

,roi_dir,genotype_base_codes,genotype_marker_fluor_codes,genotype_marker_tag_codes
0,/clusterfs/vast/abcabc/Korra_Foundation/202504...,pDQM005,tdmSG,None
1,/clusterfs/vast/abcabc/Korra_Foundation/202504...,pDQM005,tdmSG,None
2,/clusterfs/vast/abcabc/Korra_Foundation/202504...,Swinburne,Halo,None
3,/clusterfs/vast/abcabc/Korra_Foundation/202504...,Swinburne,Halo,None
4,/clusterfs/vast/abcabc/Korra_Foundation/202504...,Swinburne,Halo,None
5,/clusterfs/vast/abcabc/Korra_Foundation/202504...,Swinburne,Halo,None
6,/clusterfs/vast/abcabc/Korra_Foundation/202504...,Swinburne,Halo,None
7,/clusterfs/vast/abcabc/Korra_Foundation/202504...,Swinburne,Halo,None
8,/clusterfs/vast/abcabc/Korra_Foundation/202504...,Swinburne,Halo,None
9,/clusterfs/vast/abcabc/Korra_Foundation/202504...,Swinburne,Halo,None


In [37]:
import re
import pandas as pd

# --- build injected plasmid + RNA lookup maps --------------------------------
pl = df_inj_plasm.copy()
pl.columns = [c.strip().lower() for c in pl.columns]
pl["injected_plasmid"]   = pl["injected_plasmid"].astype(str).str.strip()
pl["plasmid_base_code"]  = pl["plasmid_base_code"].astype(str).str.strip()
inj_plasm_map = dict(zip(pl["injected_plasmid"], pl["plasmid_base_code"]))

rn = df_inj_rna.copy()
rn.columns = [c.strip().lower() for c in rn.columns]
rn["injected_rna"] = rn["injected_rna"].astype(str).str.strip()
rn_base_col = "plasmid_base_code"
rn[rn_base_col] = rn[rn_base_col].astype(str).str.strip()
inj_rna_map = dict(zip(rn["injected_rna"], rn[rn_base_col]))

def map_plasmids(text: object) -> str | None:
    if not isinstance(text, str) or not text.strip():
        return None
    toks = [t.strip() for t in re.split(r"[;,]", text) if t.strip()]
    out: list[str] = []
    for t in toks:
        if t in inj_plasm_map:
            for code in str(inj_plasm_map[t]).split(","):
                c = code.strip()
                if c:
                    out.append(c)
    return ",".join(sorted(set(out))) if out else None

def map_rnas(text: object) -> str | None:
    if not isinstance(text, str) or not text.strip():
        return None
    toks = [t.strip() for t in re.split(r"[;,]", text) if t.strip()]
    out: list[str] = []
    for t in toks:
        if t in inj_rna_map:
            for code in str(inj_rna_map[t]).split(","):
                c = code.strip()
                if c:
                    out.append(c)
    return ",".join(sorted(set(out))) if out else None

# Apply maps to sheet columns
df["treatment_plasmid_base_codes"] = df["additional plasmids injected"].apply(map_plasmids)
df["treatment_rna_base_codes"]     = df["additional mRNAs injected"].apply(map_rnas)

# --- map treatment plasmid base codes to fluor/tag markers via constructs -----
def split_codes(val: object) -> list[str]:
    if not isinstance(val, str) or not val.strip():
        return []
    return [c.strip() for c in val.split(",") if c.strip()]

texp = (
    df[["roi_dir", "treatment_plasmid_base_codes"]]
    .dropna(subset=["treatment_plasmid_base_codes"])
    .assign(code_list=lambda d: d["treatment_plasmid_base_codes"].apply(split_codes))
    .explode("code_list")
    .rename(columns={"code_list": "plasmid_base_code"})
)

texp["plasmid_base_code"] = texp["plasmid_base_code"].astype(str).str.strip()

# cons_small should come from df_constructs, e.g.:
# cons_small = df_constructs[["plasmid_code","fluor_code","tag_code","tag_pos"]].rename(
#     columns={"plasmid_code": "plasmid_base_code"}
# )

tjoin = texp.merge(cons_small, how="left", on="plasmid_base_code")

t_per_roi = (
    tjoin.groupby("roi_dir", as_index=False)
    .agg(
        treatment_marker_fluor_codes=("fluor_code", agg_uniq),
        treatment_marker_tag_codes=("tag_code", agg_uniq),
    )
)

df = df.merge(t_per_roi, how="left", on="roi_dir")

df[[
    "roi_dir",
    "treatment_plasmid_base_codes",
    "treatment_rna_base_codes",
    "treatment_marker_fluor_codes",
    "treatment_marker_tag_codes",
]].head(20)

,roi_dir,treatment_plasmid_base_codes,treatment_rna_base_codes,treatment_marker_fluor_codes,treatment_marker_tag_codes
0,/clusterfs/vast/abcabc/Korra_Foundation/202504...,None,pDQM117,NaN,NaN
1,/clusterfs/vast/abcabc/Korra_Foundation/202504...,None,pDQM117,NaN,NaN
2,/clusterfs/vast/abcabc/Korra_Foundation/202504...,pDQM140,None,mGold2s,None
3,/clusterfs/vast/abcabc/Korra_Foundation/202504...,pDQM140,None,mGold2s,None
4,/clusterfs/vast/abcabc/Korra_Foundation/202504...,pDQM140,None,mGold2s,None
5,/clusterfs/vast/abcabc/Korra_Foundation/202504...,pDQM140,None,mGold2s,None
6,/clusterfs/vast/abcabc/Korra_Foundation/202504...,pDQM140,None,mGold2s,None
7,/clusterfs/vast/abcabc/Korra_Foundation/202504...,pDQM140,None,mGold2s,None
8,/clusterfs/vast/abcabc/Korra_Foundation/202504...,pDQM140,None,mGold2s,None
9,/clusterfs/vast/abcabc/Korra_Foundation/202504...,pDQM140,None,mGold2s,None


In [38]:
from pathlib import Path

def union_markers(row):
    vals = set()
    for col in ["genotype_marker_fluor_codes", "treatment_marker_fluor_codes"]:
        v = row.get(col)
        if isinstance(v, str) and v.strip():
            for tok in v.split(","):
                tok = tok.strip()
                if tok:
                    vals.add(tok)
    return ",".join(sorted(vals)) if vals else None

df["all_marker_fluor_codes"] = df.apply(union_markers, axis=1)

cols = [
    "roi_dir", "date_experiment", "fish", "roi_name",
    "date_born", "parent_female", "parent_male",
    "genotype_base_codes", "genotype_allele_codes", "genotype_pretty",
    "genotype_marker_fluor_codes", "genotype_marker_tag_codes",
    "treatment_plasmid_base_codes", "treatment_rna_base_codes",
    "treatment_marker_fluor_codes", "treatment_marker_tag_codes",
    "all_marker_fluor_codes",
    "additional plasmids injected", "additional mRNAs injected",
    "additonal proteins injected", "additonal dye and chemicals",
    "Date born", "ZF female genotype", "ZF male genotype", "Data location",
]

cols = [c for c in cols if c in df.columns]

flat = df[cols].copy()

flat_path = WORKING / "roi_all_markers_flat_working.csv"
flat.to_csv(flat_path, index=False)

flat.shape, flat_path

((976, 25),
 PosixPath('/Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling/working/roi_all_markers_flat_working.csv'))

In [39]:
from pathlib import Path

missing = flat[(flat["parent_female"].isna()) & (flat["parent_male"].isna())].copy()
missing_path = WORKING / "roi_missing_parents_for_manual_mapping.csv"
missing.to_csv(missing_path, index=False)

missing.shape, missing_path

((172, 25),
 PosixPath('/Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling/working/roi_missing_parents_for_manual_mapping.csv'))

In [40]:
# Basic sanity checks on the flat table
print("flat shape:", flat.shape)

print("ROIs with NO parents:", ((flat["parent_female"].isna()) & (flat["parent_male"].isna())).sum())
print("ROIs with NO markers:", flat["all_marker_fluor_codes"].isna().sum())

flat.head()

flat shape: (976, 25)
ROIs with NO parents: 172
ROIs with NO markers: 175


,roi_dir,date_experiment,fish,roi_name,date_born,parent_female,parent_male,genotype_base_codes,genotype_allele_codes,genotype_pretty,...,treatment_marker_tag_codes,all_marker_fluor_codes,additional plasmids injected,additional mRNAs injected,additonal proteins injected,additonal dye and chemicals,Date born,ZF female genotype,ZF male genotype,Data location
0,/clusterfs/vast/abcabc/Korra_Foundation/202504...,20250428_mem_histone,fish1_72hpf,roi1_tail,2025-04-26,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),pDQM005,"pDQM005:301,pDQM005:301",ef1a:2xLynk:tdmSG(J) (F2 of allele 301) × ef1a...,...,NaN,tdmSG,NaN,mScarlet3-S2:H2B,NaN,NaN,2025-04-26,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),X:\abcabc\Korra_Foundation\20250428_mem_histon...
1,/clusterfs/vast/abcabc/Korra_Foundation/202504...,20250428_mem_histone,fish1_72hpf,roi2_hindbrain_spine,2025-04-26,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),pDQM005,"pDQM005:301,pDQM005:301",ef1a:2xLynk:tdmSG(J) (F2 of allele 301) × ef1a...,...,NaN,tdmSG,NaN,mScarlet3-S2:H2B,NaN,NaN,2025-04-26,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),X:\abcabc\Korra_Foundation\20250428_mem_histon...
2,/clusterfs/vast/abcabc/Korra_Foundation/202504...,20250429_mem_cytosol,fish1_24hpf,roi1,2025-04-29,membrane Halo,membrane Halo,Swinburne,"Swinburne:Swinburne,Swinburne:Swinburne",membrane Halo × membrane Halo,...,None,"Halo,mGold2s",ef1a:mGold2s,NaN,NaN,NaN,2025-04-29,membrane Halo,membrane Halo,X:\abcabc\Korra_Foundation\20250429_mem_cytoso...
3,/clusterfs/vast/abcabc/Korra_Foundation/202504...,20250429_mem_cytosol,fish1_24hpf,roi1_test,2025-04-29,membrane Halo,membrane Halo,Swinburne,"Swinburne:Swinburne,Swinburne:Swinburne",membrane Halo × membrane Halo,...,None,"Halo,mGold2s",ef1a:mGold2s,NaN,NaN,NaN,2025-04-29,membrane Halo,membrane Halo,X:\abcabc\Korra_Foundation\20250429_mem_cytoso...
4,/clusterfs/vast/abcabc/Korra_Foundation/202504...,20250429_mem_cytosol,fish2_48hpf,roi1,2025-04-29,membrane Halo,membrane Halo,Swinburne,"Swinburne:Swinburne,Swinburne:Swinburne",membrane Halo × membrane Halo,...,None,"Halo,mGold2s",ef1a:mGold2s,NaN,NaN,NaN,2025-04-29,membrane Halo,membrane Halo,X:\abcabc\Korra_Foundation\20250429_mem_cytoso...


In [42]:
from pathlib import Path
import pandas as pd


missing_path = WORKING / "roi_missing_parents_for_manual_mapping.csv"
df_missing   = pd.read_csv(missing_path)

print("df_missing columns:", df_missing.columns.tolist())

# If manual columns don't exist yet, create them as empty
for col in ["parent_female_manual", "parent_male_manual"]:
    if col not in df_missing.columns:
        df_missing[col] = None

df_missing_sub = df_missing[["roi_dir", "parent_female_manual", "parent_male_manual"]].copy()

# merge onto the main df (roi_dir is the key)
df = df.merge(df_missing_sub, how="left", on="roi_dir")

def choose_parent(manual, original):
    if isinstance(manual, str) and manual.strip():
        return manual.strip()
    return original

df["parent_female_final"] = [
    choose_parent(m, o)
    for m, o in zip(df["parent_female_manual"], df["parent_female"])
]
df["parent_male_final"] = [
    choose_parent(m, o)
    for m, o in zip(df["parent_male_manual"], df["parent_male"])
]

df[["roi_dir", "parent_female", "parent_female_manual", "parent_female_final"]].head(10)

df_missing columns: ['roi_dir', 'date_experiment', 'fish', 'roi_name', 'date_born', 'parent_female', 'parent_male', 'genotype_base_codes', 'genotype_allele_codes', 'genotype_pretty', 'genotype_marker_fluor_codes', 'genotype_marker_tag_codes', 'treatment_plasmid_base_codes', 'treatment_rna_base_codes', 'treatment_marker_fluor_codes', 'treatment_marker_tag_codes', 'all_marker_fluor_codes', 'additional plasmids injected', 'additional mRNAs injected', 'additonal proteins injected', 'additonal dye and chemicals', 'Date born', 'ZF female genotype', 'ZF male genotype', 'Data location']


,roi_dir,parent_female,parent_female_manual,parent_female_final
0,/clusterfs/vast/abcabc/Korra_Foundation/202504...,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),NaN,ef1a:2xLynk:tdmSG(J) (F2 of allele 301)
1,/clusterfs/vast/abcabc/Korra_Foundation/202504...,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),NaN,ef1a:2xLynk:tdmSG(J) (F2 of allele 301)
2,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,NaN,membrane Halo
3,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,NaN,membrane Halo
4,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,NaN,membrane Halo
5,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,NaN,membrane Halo
6,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,NaN,membrane Halo
7,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,NaN,membrane Halo
8,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,NaN,membrane Halo
9,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,NaN,membrane Halo


In [51]:
from pathlib import Path
import pandas as pd


# Start from the flat table we’ve already built
flat = flat_auto.copy()

# 1) Make sure parent_female / parent_male are populated from the *_final columns
flat["parent_female"] = flat.get("parent_female_final", flat.get("parent_female"))
flat["parent_male"]   = flat.get("parent_male_final", flat.get("parent_male"))

# 2) Core annotation columns the loader / view care about
annot_cols = [
    "roi_dir",
    "parent_female", "parent_male",
    "genotype_pretty_final",        # will be renamed → genotype_pretty
    "genotype_base_codes", "genotype_allele_codes",
    "genotype_marker_fluor_codes", "genotype_marker_tag_codes",
    "treatment_plasmid_base_codes", "treatment_rna_base_codes",
    "treatment_marker_fluor_codes", "treatment_marker_tag_codes",
    "all_marker_fluor_codes",
]

# include dyes if present
if "treatment_dye_base_codes" in flat.columns:
    annot_cols.append("treatment_dye_base_codes")

# 3) Also carry the synthetic plate/slot IDs if they exist in flat_auto
#    (these won't be used by the current DB table yet, but we preserve them in the CSV)
for extra_col in ["plate_id_filled", "slot_id_filled"]:
    if extra_col in flat.columns:
        annot_cols.append(extra_col)

# keep only columns that actually exist
annot_cols = [c for c in annot_cols if c in flat.columns]

annot = flat[annot_cols].copy()

# 4) Rename genotype_pretty_final → genotype_pretty for the DB schema
annot = annot.rename(columns={"genotype_pretty_final": "genotype_pretty"})

annot_path = WORKING / "imaging_roi_annotations_AUTO.csv"
annot.to_csv(annot_path, index=False)

annot.shape, annot_path

((976, 13),
 PosixPath('/Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling/working/imaging_roi_annotations_AUTO.csv'))

In [44]:
from pathlib import Path
import pandas as pd



def union_markers(row: pd.Series) -> str | None:
    vals = set()
    for col in ["genotype_marker_fluor_codes", "treatment_marker_fluor_codes"]:
        v = row.get(col)
        if isinstance(v, str) and v.strip():
            for tok in v.split(","):
                tok = tok.strip()
                if tok:
                    vals.add(tok)
    return ",".join(sorted(vals)) if vals else None

df["all_marker_fluor_codes"] = df.apply(union_markers, axis=1)

cols_auto = [
    "roi_dir", "date_experiment", "fish", "roi_name",
    "date_born",
    "parent_female", "parent_male",
    "genotype_base_codes", "genotype_allele_codes", "genotype_pretty",
    "genotype_marker_fluor_codes", "genotype_marker_tag_codes",
    "treatment_plasmid_base_codes", "treatment_rna_base_codes",
    "treatment_marker_fluor_codes", "treatment_marker_tag_codes",
    "all_marker_fluor_codes",
    "additional plasmids injected", "additional mRNAs injected",
    "additonal proteins injected", "additonal dye and chemicals",
    "Date born", "ZF female genotype", "ZF male genotype", "Data location",
]

cols_auto = [c for c in cols_auto if c in df.columns]

flat_auto = df[cols_auto].copy()

flat_auto_path = WORKING / "roi_all_markers_flat_AUTO.csv"
flat_auto.to_csv(flat_auto_path, index=False)

flat_auto.shape, flat_auto_path

((976, 25),
 PosixPath('/Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling/working/roi_all_markers_flat_AUTO.csv'))

In [45]:
def union_markers(row):
    vals = set()
    for col in ["genotype_marker_fluor_codes", "treatment_marker_fluor_codes"]:
        v = row.get(col)
        if isinstance(v, str) and v.strip():
            for tok in v.split(","):
                tok = tok.strip()
                if tok:
                    vals.add(tok)
    return ",".join(sorted(vals)) if vals else None

df["all_marker_fluor_codes"] = df.apply(union_markers, axis=1)

cols_final = [
    "roi_dir", "date_experiment", "fish", "roi_name",
    "date_born",
    "parent_female_final", "parent_male_final",
    "genotype_base_codes", "genotype_allele_codes", "genotype_pretty_final",
    "genotype_marker_fluor_codes", "genotype_marker_tag_codes",
    "treatment_plasmid_base_codes", "treatment_rna_base_codes",
    "treatment_marker_fluor_codes", "treatment_marker_tag_codes",
    "all_marker_fluor_codes",
    "additional plasmids injected", "additional mRNAs injected",
    "additonal proteins injected", "additonal dye and chemicals",
    "Date born", "ZF female genotype", "ZF male genotype", "Data location",
]

cols_final = [c for c in cols_final if c in df.columns]

flat_complete = df[cols_final].copy()

flat_complete_path = WORKING / "roi_all_markers_flat_COMPLETE.csv"
flat_complete.to_csv(flat_complete_path, index=False)

flat_complete.shape, flat_complete_path

((976, 24),
 PosixPath('/Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling/working/roi_all_markers_flat_COMPLETE.csv'))

In [46]:
print("Total ROIs (rows):", len(flat_auto))
print("Missing parents (both female/male NaN):",
      ((flat_auto["parent_female"].isna()) & (flat_auto["parent_male"].isna())).sum())
print("Missing all_marker_fluor_codes:",
      flat_auto["all_marker_fluor_codes"].isna().sum())

Total ROIs (rows): 976
Missing parents (both female/male NaN): 172
Missing all_marker_fluor_codes: 175


In [52]:
# === CELL: build imaging_roi_annotations_AUTO.csv for DB import ===
from pathlib import Path
import pandas as pd



# Start from the flat ROI+markers table we already built
flat = flat_auto.copy()

# 1) Use the best-available parents (manual or original)
flat["parent_female"] = flat.get("parent_female_final", flat.get("parent_female"))
flat["parent_male"]   = flat.get("parent_male_final",   flat.get("parent_male"))

# 2) Columns we want to send into imaging_roi_annotations
annot_cols = [
    "roi_dir",
    "parent_female", "parent_male",
    "genotype_pretty_final",        # will be renamed -> genotype_pretty
    "genotype_base_codes", "genotype_allele_codes",
    "genotype_marker_fluor_codes", "genotype_marker_tag_codes",
    "treatment_plasmid_base_codes", "treatment_rna_base_codes",
    "treatment_marker_fluor_codes", "treatment_marker_tag_codes",
    "all_marker_fluor_codes",
]

# include dye base codes if we have them
if "treatment_dye_base_codes" in flat.columns:
    annot_cols.append("treatment_dye_base_codes")

# include synthetic plate/slot IDs if present
for extra_col in ["plate_id_filled", "slot_id_filled"]:
    if extra_col in flat.columns:
        annot_cols.append(extra_col)

# keep only columns that actually exist in flat
annot_cols = [c for c in annot_cols if c in flat.columns]

annot = flat[annot_cols].copy()

# 3) Normalize column names to what the DB + loader expect
annot = annot.rename(
    columns={
        "genotype_pretty_final": "genotype_pretty",
    }
)

annot_path = WORKING / "imaging_roi_annotations_AUTO.csv"
annot.to_csv(annot_path, index=False)

print("annotations shape:", annot.shape)
print("wrote:", annot_path)
annot.head(5)

annotations shape: (976, 13)
wrote: /Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling/working/imaging_roi_annotations_AUTO.csv


,roi_dir,parent_female,parent_male,genotype_pretty,genotype_base_codes,genotype_allele_codes,genotype_marker_fluor_codes,genotype_marker_tag_codes,treatment_plasmid_base_codes,treatment_rna_base_codes,treatment_marker_fluor_codes,treatment_marker_tag_codes,all_marker_fluor_codes
0,/clusterfs/vast/abcabc/Korra_Foundation/202504...,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301) × ef1a...,pDQM005,"pDQM005:301,pDQM005:301",tdmSG,None,None,pDQM117,NaN,NaN,tdmSG
1,/clusterfs/vast/abcabc/Korra_Foundation/202504...,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301) × ef1a...,pDQM005,"pDQM005:301,pDQM005:301",tdmSG,None,None,pDQM117,NaN,NaN,tdmSG
2,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,membrane Halo,membrane Halo × membrane Halo,Swinburne,"Swinburne:Swinburne,Swinburne:Swinburne",Halo,None,pDQM140,None,mGold2s,None,"Halo,mGold2s"
3,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,membrane Halo,membrane Halo × membrane Halo,Swinburne,"Swinburne:Swinburne,Swinburne:Swinburne",Halo,None,pDQM140,None,mGold2s,None,"Halo,mGold2s"
4,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,membrane Halo,membrane Halo × membrane Halo,Swinburne,"Swinburne:Swinburne,Swinburne:Swinburne",Halo,None,pDQM140,None,mGold2s,None,"Halo,mGold2s"


In [48]:
import pandas as pd
from pathlib import Path

BASE    = Path("/Users/davekokel/Projects/carp_v2")
WORKING = BASE / "seed_kits" / "legacy_wrangling" / "working"
WORKING.mkdir(parents=True, exist_ok=True)

def _norm_parent(val) -> str:
    if isinstance(val, str):
        return val.strip()
    if pd.isna(val):
        return ""
    return str(val).strip()

def make_genotype_pretty_final(row: pd.Series) -> str | None:
    f = _norm_parent(row.get("parent_female_final", row.get("parent_female")))
    m = _norm_parent(row.get("parent_male_final", row.get("parent_male")))
    if f and m:
        return f"{f} × {m}"
    if f:
        return f
    if m:
        return m
    return None

df["genotype_pretty_final"] = df.apply(make_genotype_pretty_final, axis=1)

def union_markers(row: pd.Series) -> str | None:
    vals = set()
    for col in ["genotype_marker_fluor_codes", "treatment_marker_fluor_codes"]:
        v = row.get(col)
        if isinstance(v, str) and v.strip():
            for tok in v.split(","):
                tok = tok.strip()
                if tok:
                    vals.add(tok)
    return ",".join(sorted(vals)) if vals else None

df["all_marker_fluor_codes"] = df.apply(union_markers, axis=1)

df[[
    "roi_dir",
    "parent_female_final", "parent_male_final",
    "genotype_base_codes", "genotype_allele_codes",
    "genotype_pretty_final",
    "all_marker_fluor_codes",
]].head(10)


,roi_dir,parent_female_final,parent_male_final,genotype_base_codes,genotype_allele_codes,genotype_pretty_final,all_marker_fluor_codes
0,/clusterfs/vast/abcabc/Korra_Foundation/202504...,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),pDQM005,"pDQM005:301,pDQM005:301",ef1a:2xLynk:tdmSG(J) (F2 of allele 301) × ef1a...,tdmSG
1,/clusterfs/vast/abcabc/Korra_Foundation/202504...,ef1a:2xLynk:tdmSG(J) (F2 of allele 301),ef1a:2xLynk:tdmSG(J) (F2 of allele 301),pDQM005,"pDQM005:301,pDQM005:301",ef1a:2xLynk:tdmSG(J) (F2 of allele 301) × ef1a...,tdmSG
2,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,membrane Halo,Swinburne,"Swinburne:Swinburne,Swinburne:Swinburne",membrane Halo × membrane Halo,"Halo,mGold2s"
3,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,membrane Halo,Swinburne,"Swinburne:Swinburne,Swinburne:Swinburne",membrane Halo × membrane Halo,"Halo,mGold2s"
4,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,membrane Halo,Swinburne,"Swinburne:Swinburne,Swinburne:Swinburne",membrane Halo × membrane Halo,"Halo,mGold2s"
5,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,membrane Halo,Swinburne,"Swinburne:Swinburne,Swinburne:Swinburne",membrane Halo × membrane Halo,"Halo,mGold2s"
6,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,membrane Halo,Swinburne,"Swinburne:Swinburne,Swinburne:Swinburne",membrane Halo × membrane Halo,"Halo,mGold2s"
7,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,membrane Halo,Swinburne,"Swinburne:Swinburne,Swinburne:Swinburne",membrane Halo × membrane Halo,"Halo,mGold2s"
8,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,membrane Halo,Swinburne,"Swinburne:Swinburne,Swinburne:Swinburne",membrane Halo × membrane Halo,"Halo,mGold2s"
9,/clusterfs/vast/abcabc/Korra_Foundation/202504...,membrane Halo,membrane Halo,Swinburne,"Swinburne:Swinburne,Swinburne:Swinburne",membrane Halo × membrane Halo,"Halo,mGold2s"


In [49]:
cols_auto = [
    "roi_dir", "date_experiment", "fish", "roi_name",
    "date_born",
    "parent_female_final", "parent_male_final",
    "genotype_base_codes", "genotype_allele_codes", "genotype_pretty_final",
    "genotype_marker_fluor_codes", "genotype_marker_tag_codes",
    "treatment_plasmid_base_codes", "treatment_rna_base_codes",
    "treatment_marker_fluor_codes", "treatment_marker_tag_codes",
    "all_marker_fluor_codes",
    "additional plasmids injected", "additional mRNAs injected",
    "additonal proteins injected", "additonal dye and chemicals",
    "Date born", "ZF female genotype", "ZF male genotype", "Data location",
]

cols_auto = [c for c in cols_auto if c in df.columns]

flat_auto = df[cols_auto].copy()
flat_auto_path = WORKING / "roi_all_markers_flat_AUTO.csv"
flat_auto.to_csv(flat_auto_path, index=False)

flat_auto.shape, flat_auto_path

((976, 25),
 PosixPath('/Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling/working/roi_all_markers_flat_AUTO.csv'))

In [50]:
annot_cols = [
    "roi_dir",
    "parent_female_final", "parent_male_final",
    "genotype_pretty_final",
    "genotype_base_codes", "genotype_allele_codes",
    "genotype_marker_fluor_codes", "genotype_marker_tag_codes",
    "treatment_plasmid_base_codes", "treatment_rna_base_codes",
    "treatment_marker_fluor_codes", "treatment_marker_tag_codes",
    "all_marker_fluor_codes",
]

annot_cols = [c for c in annot_cols if c in flat_auto.columns]

annot = flat_auto[annot_cols].copy()
annot_path = WORKING / "imaging_roi_annotations_AUTO.csv"
annot.to_csv(annot_path, index=False)

annot.shape, annot_path

((976, 13),
 PosixPath('/Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling/working/imaging_roi_annotations_AUTO.csv'))

In [59]:
# ----------------------------------------------------------
# Build imaging_clutch_memberships_from_sheet_draft.csv
# ----------------------------------------------------------

# Work on a copy so we don't mutate df_sheet
sheet_members = df_sheet.copy()

# Locate the "Data location" column robustly
data_loc_col = None
for c in sheet_members.columns:
    if str(c).strip().lower().startswith("data location") or str(c).strip().lower().startswith("data_location"):
        data_loc_col = c
        break

if data_loc_col is None:
    raise ValueError("Could not find a 'Data location' column in df_sheet")

# Drop rows without a data location (no mount)
sheet_members = sheet_members[~sheet_members[data_loc_col].isna()].copy()

# Add sheet_row_index (1-based Excel row index: header is row 1, data starts at row 2)
sheet_members = sheet_members.reset_index(drop=True)
sheet_members["sheet_row_index"] = sheet_members.index + 2

# Normalize date_mount and build YYYYMMDD string
sheet_members["date_mount"] = pd.to_datetime(sheet_members["date_mount"]).dt.date
sheet_members["date_mount_yyyymmdd"] = pd.to_datetime(sheet_members["date_mount"]).dt.strftime("%Y%m%d")

def last_folder(path: str) -> str | None:
    if not isinstance(path, str):
        return None
    p = path.replace("\\", "/")
    if ":" in p:
        p = p.split(":", 1)[1]
    p = p.lstrip("/")
    return p.split("/")[-1]

# mount_key = last folder of Data location (Option A)
sheet_members["mount_key"] = sheet_members[data_loc_col].apply(last_folder)

# Sort by date_mount then sheet_row_index to define per-date slot order
sheet_members = sheet_members.sort_values(["date_mount", "sheet_row_index"]).reset_index(drop=True)

# slot_index_global: index within each date_mount in sheet order (1-based)
sheet_members["slot_index_global"] = sheet_members.groupby("date_mount").cumcount() + 1

# 6 slots per plate: compute plateN and slotM
sheet_members["plateN"] = ((sheet_members["slot_index_global"] - 1) // 6) + 1
sheet_members["slotM"] = ((sheet_members["slot_index_global"] - 1) % 6) + 1

# Build plate_id_filled and slot_id_filled
sheet_members["plate_id_filled"] = (
    sheet_members["date_mount_yyyymmdd"] + "-plate" + sheet_members["plateN"].astype(str)
)
sheet_members["slot_id_filled"] = (
    sheet_members["date_mount_yyyymmdd"]
    + "-plate"
    + sheet_members["plateN"].astype(str)
    + "-slot"
    + sheet_members["slotM"].astype(str)
)

# Select clean output columns
cols_out = [
    "date_mount",
    "date_mount_yyyymmdd",
    "sheet_row_index",
]

if "mount_id" in sheet_members.columns:
    cols_out.append("mount_id")

cols_out.extend(
    [
        data_loc_col,
        "mount_key",
        "slot_index_global",
        "plateN",
        "slotM",
        "plate_id_filled",
        "slot_id_filled",
    ]
)

membership_df = sheet_members[cols_out].copy()

WORKING.mkdir(parents=True, exist_ok=True)
membership_path = WORKING / "imaging_clutch_memberships_from_sheet_draft.csv"
membership_df.to_csv(membership_path, index=False)

membership_df.shape, membership_path

((231, 11),
 PosixPath('/Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling/working/imaging_clutch_memberships_from_sheet_draft.csv'))

In [61]:
# ----------------------------------------------------------
# Attach plate/slot to each ROI and build roi_code
# ----------------------------------------------------------
from pathlib import Path
import pandas as pd

# Load membership table we just generated
membership_path = WORKING / "imaging_clutch_memberships_from_sheet_draft.csv"
df_membership = pd.read_csv(membership_path)

print("df_membership:", df_membership.shape)

# Heuristic: find the ROI directory column in df_roi
roi_dir_col = None
for c in df_roi.columns:
    cl = str(c).lower()
    if "roi" in cl and "dir" in cl:
        roi_dir_col = c
        break
    if "root" in cl and "tiff" in cl:
        roi_dir_col = c
        break

if roi_dir_col is None:
    raise ValueError(
        f"Could not infer ROI directory column in df_roi; "
        f"available columns: {list(df_roi.columns)}"
    )

print("Using ROI directory column:", roi_dir_col)

def last_folder(path):
    if not isinstance(path, str):
        return None
    p = path.replace("\\", "/")
    if ":" in p:
        p = p.split(":", 1)[1]
    p = p.lstrip("/")
    return p.split("/")[-1]

# mount_key from ROI dir, matching the Option A logic we used on the sheet
df_roi_slots = df_roi.copy()
df_roi_slots["mount_key"] = df_roi_slots[roi_dir_col].apply(last_folder)

# Sanity: keep only columns we need from membership
mem_cols = [
    "mount_key",
    "date_mount",
    "date_mount_yyyymmdd",
    "plate_id_filled",
    "slot_id_filled",
]

# One row per mount_key: take the first occurrence in sheet order
df_membership_key = (
    df_membership[mem_cols]
    .sort_values(["date_mount", "date_mount_yyyymmdd"])
    .drop_duplicates(subset=["mount_key"], keep="first")
)

print("df_membership_key shape:", df_membership_key.shape)
print("unique mount_keys:", df_membership_key["mount_key"].nunique())

# Join ROI → membership via mount_key
df_roi_slots = df_roi_slots.merge(
    df_membership_key,
    on="mount_key",
    how="left",
    validate="m:1",
)

# Sort within each slot for stable roi numbering
df_roi_slots = df_roi_slots.sort_values(
    ["plate_id_filled", "slot_id_filled", roi_dir_col]
).reset_index(drop=True)

# roi_index_within_slot: index within each slot (1-based)
df_roi_slots["roi_index_within_slot"] = (
    df_roi_slots.groupby(["plate_id_filled", "slot_id_filled"])
    .cumcount()
    .astype("Int64") + 1
)

# Build roi_code = {slot_id_filled}-roiNN
# Leave roi_code as NA if slot_id_filled is missing (no membership match)
def make_roi_code(row):
    slot = row["slot_id_filled"]
    idx  = row["roi_index_within_slot"]
    if pd.isna(slot) or pd.isna(idx):
        return None
    return f"{slot}-roi{int(idx):02d}"

df_roi_slots["roi_code"] = df_roi_slots.apply(make_roi_code, axis=1)

# Save a working CSV with plate/slot/roi_code attached to each ROI
roi_slots_path = WORKING / "roi_with_plate_slot_and_code_working.csv"
df_roi_slots.to_csv(roi_slots_path, index=False)

df_roi_slots.shape, roi_slots_path

df_membership: (231, 11)
Using ROI directory column: roi_dir
df_membership_key shape: (169, 5)
unique mount_keys: 168


((976, 14),
 PosixPath('/Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling/working/roi_with_plate_slot_and_code_working.csv'))

In [65]:
# ----------------------------------------------------------
# Build imaging_roi_annotations_AUTO.csv from flat + membership
# ----------------------------------------------------------
import pandas as pd

flat_path = WORKING / "roi_all_markers_flat_working.csv"
df_flat = pd.read_csv(flat_path)

membership_path = WORKING / "imaging_clutch_memberships_from_sheet_draft.csv"
df_membership = pd.read_csv(membership_path)

print("df_flat:", df_flat.shape)
print("df_membership:", df_membership.shape)

# 1) Make sure membership is unique per Data location
#    (take the first occurrence in sheet order)
df_membership = df_membership.sort_values(["date_mount", "sheet_row_index"])
df_membership_unique = df_membership.drop_duplicates(
    subset=["Data location"], keep="first"
)

print("df_membership_unique:", df_membership_unique.shape)

# 2) Merge plate/slot into the flat ROI table via *Data location*
df_ann = df_flat.merge(
    df_membership_unique[["Data location", "plate_id_filled", "slot_id_filled"]],
    on="Data location",
    how="left",
)

print(
    "Total ROIs:", len(df_ann),
    "  with plate/slot:", df_ann["plate_id_filled"].notna().sum()
)

# 3) Within each (plate, slot), assign roi_index_within_slot and roi_code
df_ann = df_ann.sort_values(
    ["plate_id_filled", "slot_id_filled", "roi_dir", "roi_name"]
).reset_index(drop=True)

def assign_roi_index(group: pd.DataFrame) -> pd.DataFrame:
    if group["plate_id_filled"].isna().all():
        group["roi_index_within_slot"] = pd.NA
    else:
        group["roi_index_within_slot"] = (
            pd.Series(range(1, len(group) + 1), index=group.index).astype("Int64")
        )
    return group

df_ann = (
    df_ann.groupby(
        ["plate_id_filled", "slot_id_filled"],
        dropna=False,
        group_keys=False,
        sort=False,
    )
    .apply(assign_roi_index, include_groups=True)
)

def make_roi_code(row):
    slot = row["slot_id_filled"]
    idx  = row["roi_index_within_slot"]
    if pd.isna(slot) or pd.isna(idx):
        return None
    return f"{slot}-roi{int(idx):02d}"

df_ann["roi_code"] = df_ann.apply(make_roi_code, axis=1)

# 4) Write out the canonical auto-annotations CSV
out_path = WORKING / "imaging_roi_annotations_AUTO.csv"
df_ann.to_csv(out_path, index=False)

out_path, df_ann["plate_id_filled"].notna().sum()

df_flat: (976, 25)
df_membership: (231, 11)
df_membership_unique: (219, 11)
Total ROIs: 976   with plate/slot: 825


/var/folders/29/cdrb2nrn01s4d1_n6gvxy5j80000gn/T/ipykernel_55223/3774184756.py:57: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  .apply(assign_roi_index, include_groups=True)
/var/folders/29/cdrb2nrn01s4d1_n6gvxy5j80000gn/T/ipykernel_55223/3774184756.py:57: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(assign_roi_index, include_groups=True)


(PosixPath('/Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling/working/imaging_roi_annotations_AUTO.csv'),
 np.int64(825))

In [66]:
# ----------------------------------------------------------
# Inspect ROI path patterns, especially for missing roi_code
# ----------------------------------------------------------
import pandas as pd

flat_path = WORKING / "roi_all_markers_flat_working.csv"
df_flat = pd.read_csv(flat_path)

auto_path = WORKING / "imaging_roi_annotations_AUTO.csv"
df_ann = pd.read_csv(auto_path)

print("df_flat:", df_flat.shape)
print("df_ann:", df_ann.shape)

# Focus on rows that *do not* have roi_code yet
missing = df_ann[df_ann["roi_code"].isna()].copy()
print("Missing roi_code rows:", len(missing))

# Join back the flat metadata so we can see roi_dir etc. reliably
missing = missing.merge(
    df_flat[["roi_dir", "date_experiment", "fish", "roi_name"]],
    on=["roi_dir", "date_experiment", "fish", "roi_name"],
    how="left",
)

def norm_path(p: str) -> str | None:
    if not isinstance(p, str):
        return None
    p = p.replace("\\", "/")
    p = p.rstrip("/")
    return p

missing["roi_dir_norm"] = missing["roi_dir"].apply(norm_path)
missing["segments"] = missing["roi_dir_norm"].str.split("/")
missing["depth"] = missing["segments"].str.len()

print("\nPath depth value_counts for missing roi_code:")
print(missing["depth"].value_counts())

# Show some example segment tails per depth, to see patterns
def show_examples_for_depth(d, n=5):
    subset = missing[missing["depth"] == d].copy()
    print("\n=== Depth", d, "examples (up to", n, ") ===")
    for _, row in subset.head(n).iterrows():
        segs = row["segments"]
        tail = segs[-6:] if len(segs) > 6 else segs
        print("... /" + "/".join(tail))

for d in sorted(missing["depth"].dropna().unique()):
    show_examples_for_depth(d, n=8)

df_flat: (976, 25)
df_ann: (976, 29)
Missing roi_code rows: 151

Path depth value_counts for missing roi_code:
depth
7    93
8    58
Name: count, dtype: int64

=== Depth 7 examples (up to 8 ) ===
... /clusterfs/vast/abcabc/Aang_Foundation/20250721_72hpf_mrna_mSG_organelle_LLS-SIM/er_roi1
... /clusterfs/vast/abcabc/Aang_Foundation/20250721_72hpf_mrna_mSG_organelle_LLS-SIM/er_roi2
... /clusterfs/vast/abcabc/Aang_Foundation/20250721_72hpf_mrna_mSG_organelle_LLS-SIM/er_roi3
... /clusterfs/vast/abcabc/Aang_Foundation/20250721_72hpf_mrna_mSG_organelle_LLS-SIM/mito_roi1
... /clusterfs/vast/abcabc/Aang_Foundation/20250721_72hpf_mrna_mSG_organelle_LLS-SIM/mito_roi2
... /clusterfs/vast/abcabc/Aang_Foundation/20250723_er-mSG_mem-mChilada/fish1_24hpf_roi1
... /clusterfs/vast/abcabc/Aang_Foundation/20251002_mem_mito/fish10_24hpf_roi1
... /clusterfs/vast/abcabc/Aang_Foundation/20251002_mem_mito/fish10_24hpf_roi2

=== Depth 8 examples (up to 8 ) ===
... /vast/abcabc/Aang_Foundation/Denoising/20251028

In [67]:
# ----------------------------------------------------------
# Fallback: infer plate/slot/roi_code from roi_dir paths
# for ROIs that still have no plate/slot
# ----------------------------------------------------------
import re
import pandas as pd
import numpy as np

auto_path = WORKING / "imaging_roi_annotations_AUTO.csv"
df_ann = pd.read_csv(auto_path)

print("Before fallback, roi_code NA:",
      df_ann["roi_code"].isna().sum())

# Work only on rows that are missing plate/slot (and thus roi_code)
missing = df_ann[df_ann["plate_id_filled"].isna()].copy()
print("Rows missing plate/slot:", len(missing))

def norm_path(p: str) -> str | None:
    if not isinstance(p, str):
        return None
    return p.replace("\\", "/").rstrip("/")

missing["roi_dir_norm"] = missing["roi_dir"].apply(norm_path)
missing["segments"] = missing["roi_dir_norm"].str.split("/")

# ---- infer a YYYYMMDD date string ("implied_date") ----
def extract_date_from_experiment(date_experiment, segments):
    if isinstance(date_experiment, str) and len(date_experiment) >= 8 and date_experiment[:8].isdigit():
        return date_experiment[:8]
    if isinstance(segments, list):
        for seg in reversed(segments):
            if isinstance(seg, str) and len(seg) >= 8 and seg[:8].isdigit():
                return seg[:8]
    return None

missing["implied_date"] = [
    extract_date_from_experiment(de, segs)
    for de, segs in zip(missing["date_experiment"], missing["segments"])
]

print("Missing implied_date:",
      missing["implied_date"].isna().sum())

# Drop the truly un-parsable one (e.g. analysis_test)
missing2 = missing[missing["implied_date"].notna()].copy()

# ---- infer a "fish_key" for slot grouping ----
def infer_fish_key(segments):
    if not isinstance(segments, list) or not segments:
        return None
    last = segments[-1]
    parent = segments[-2] if len(segments) >= 2 else None

    # Prefer a separate /fishX/ folder if present
    if isinstance(parent, str) and parent.lower().startswith("fish"):
        return parent

    # Otherwise, try pattern like "fish1_24hpf_roi3" or "er_roi2"
    if isinstance(last, str):
        m = re.search(r"(.+?)_roi\d+$", last)
        if m:
            return m.group(1)

    # Fallback: use the parent segment (e.g. "20251028_peroxi")
    return parent

missing2["fish_key"] = missing2["segments"].apply(infer_fish_key)

# ---- assign slot numbers per date based on fish_key ----
slot_map = {}
for date, sub in missing2.groupby("implied_date"):
    fish_keys = sorted(set(sub["fish_key"]))
    slot_map[date] = {fk: i + 1 for i, fk in enumerate(fish_keys)}

# Build fallback plate/slot ids
def assign_plate_slot(row):
    date = row["implied_date"]
    fk = row["fish_key"]
    if pd.isna(date) or pd.isna(fk):
        return pd.Series(
            {
                "plate_id_filled_fb": np.nan,
                "slot_id_filled_fb": np.nan,
            }
        )
    slot_index = slot_map[date].get(fk)
    plate = f"{date}-plate1"
    slot_id = (
        f"{date}-plate1-slot{slot_index}"
        if slot_index is not None
        else np.nan
    )
    return pd.Series(
        {
            "plate_id_filled_fb": plate,
            "slot_id_filled_fb": slot_id,
        }
    )

fb_ids = missing2.apply(assign_plate_slot, axis=1)
missing2 = pd.concat([missing2, fb_ids], axis=1)

# ---- assign roi_index_within_slot and roi_code ----
missing2 = missing2.sort_values(
    ["plate_id_filled_fb", "slot_id_filled_fb", "roi_dir_norm", "roi_name"]
).reset_index()  # keep original index in a column

missing2["roi_index_fb"] = (
    missing2.groupby(
        ["plate_id_filled_fb", "slot_id_filled_fb"]
    ).cumcount()
    + 1
)

missing2["roi_code_fb"] = missing2.apply(
    lambda r: (
        f"{r['slot_id_filled_fb']}-roi{int(r['roi_index_fb']):02d}"
        if isinstance(r["slot_id_filled_fb"], str)
        else None
    ),
    axis=1,
)

print("Fallback rows with roi_code_fb:",
      missing2["roi_code_fb"].notna().sum())

# ---- write fallback values back into df_ann ----
idx_map = missing2.set_index("index")[
    ["plate_id_filled_fb", "slot_id_filled_fb", "roi_index_fb", "roi_code_fb"]
]

for src, dst in [
    ("plate_id_filled_fb", "plate_id_filled"),
    ("slot_id_filled_fb", "slot_id_filled"),
    ("roi_index_fb", "roi_index_within_slot"),
    ("roi_code_fb", "roi_code"),
]:
    s = idx_map[src]
    df_ann.loc[s.index, dst] = s

print("After fallback, roi_code NA:",
      df_ann["roi_code"].isna().sum())

# Overwrite the AUTO CSV with the enriched version
df_ann.to_csv(auto_path, index=False)

auto_path

Before fallback, roi_code NA: 151
Rows missing plate/slot: 151
Missing implied_date: 1
Fallback rows with roi_code_fb: 150
After fallback, roi_code NA: 1


PosixPath('/Users/davekokel/Projects/carp_v2/seed_kits/legacy_wrangling/working/imaging_roi_annotations_AUTO.csv')